In [45]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv


In [46]:
# look for the encoding 
import subprocess
import chardet

subprocess.run(["pip", "install", "chardet"], capture_output=True)

with open("/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv", "rb") as f:
    raw = f.read(50000) 

result = chardet.detect(raw)
print(result)


{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [47]:
df = pd.read_csv(
    '/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv', encoding="Windows-1252"
)
list(df.columns)


['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [48]:
# quality checks
print("------ data frame dimension---------------\n", df.shape)
print("------ data frame type---------------------\n", df.dtypes)
print("------ data null values-------------------- \n",df.isnull().sum())
print("------- Numerical values------------------- \n", 
      df[["Sales", "Profit", "Discount", "Quantity"]].describe())

------ data frame dimension---------------
 (9994, 21)
------ data frame type---------------------
 Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object
------ data null values-------------------- 
 Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Nam

In [49]:
from io import StringIO

# rewrite the name columns

with open("/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv", "rb") as f:
    raw = f.read()

# Remove null bytes and decode
cleaned = raw.replace(b"\x00", b"")
decoded = cleaned.decode("utf-8", errors="ignore")

# Feed the cleaned string directly to pandas
df = pd.read_csv(StringIO(decoded))

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex = False)
    .str.replace("-", "_", regex = False)
)

print(f"Shape: {df.shape}")
print(df.columns.tolist())

Shape: (9994, 21)
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [50]:
import sqlite3
import os

# reset the database if not exist
if os.path.exists("superstore.db"):
    conn.close()
    os.remove("superstore.db")

conn = sqlite3.connect("superstore.db")
df.to_sql("orders_raw", conn, if_exists="replace", index=False)

print("Raw table loaded:", pd.read_sql("SELECT COUNT(*) AS rows FROM orders_raw", conn).iloc[0,0], "rows")

Raw table loaded: 9994 rows


In [51]:
# create the four normalized tables 

conn.executescript("""

    -- 1. CUSTOMERS
    CREATE TABLE IF NOT EXISTS customers AS
    SELECT DISTINCT
        customer_id,
        customer_name,
        segment
    FROM orders_raw;

    -- 2. PRODUCTS
    CREATE TABLE IF NOT EXISTS products AS
    SELECT DISTINCT
        product_id,
        product_name,
        category,
        sub_category
    FROM orders_raw;

    -- 3. GEOGRAPHY
    CREATE TABLE IF NOT EXISTS geography AS
    SELECT DISTINCT
        postal_code,
        city,
        state,
        region,
        country
    FROM orders_raw;

    -- 4. ORDER_ITEMS (fact table)
    CREATE TABLE IF NOT EXISTS order_items AS
    SELECT
        row_id,
        order_id,
        customer_id,
        product_id,
        postal_code,
        order_date,
        ship_date,
        ship_mode,
        quantity,
        sales,
        discount,
        profit
    FROM orders_raw;

""")

conn.commit()
print("Normalization complete.")

Normalization complete.


In [52]:
tables = ["customers", "products", "geography", "order_items"]

for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
    cols  = pd.read_sql(f"SELECT * FROM {table} LIMIT 0", conn).columns.tolist()
    print(f"{table:<15} {count:>5} rows   columns: {cols}")

customers         793 rows   columns: ['customer_id', 'customer_name', 'segment']
products         1894 rows   columns: ['product_id', 'product_name', 'category', 'sub_category']
geography         632 rows   columns: ['postal_code', 'city', 'state', 'region', 'country']
order_items      9994 rows   columns: ['row_id', 'order_id', 'customer_id', 'product_id', 'postal_code', 'order_date', 'ship_date', 'ship_mode', 'quantity', 'sales', 'discount', 'profit']


In [53]:
query = """
    SELECT
        c.segment,
        p.category,
        COUNT(*)                        AS orders,
        ROUND(SUM(oi.sales), 2)         AS total_sales,
        ROUND(SUM(oi.profit), 2)        AS total_profit,
        ROUND(AVG(oi.profit / NULLIF(oi.sales, 0)) * 100, 2) AS avg_margin_pct
    FROM order_items oi
    JOIN customers  c ON oi.customer_id = c.customer_id
    JOIN products   p ON oi.product_id  = p.product_id
    GROUP BY c.segment, p.category
    ORDER BY total_profit DESC
"""

pd.read_sql(query, conn)

,segment,category,orders,total_sales,total_profit,avg_margin_pct
0,Consumer,Technology,995,436285.80,75687.24,15.27
1,Consumer,Office Supplies,3210,373176.70,57392.83,13.07
2,Corporate,Technology,574,260268.49,45206.41,15.33
3,Corporate,Office Supplies,1860,235077.73,41302.71,13.91
4,Home Office,Technology,360,197078.99,32522.06,16.29
5,Home Office,Office Supplies,1116,128494.16,27417.81,16.91
6,Consumer,Furniture,1175,409725.80,8072.76,3.34
7,Corporate,Furniture,657,230677.44,7857.41,4.82
8,Home Office,Furniture,384,123881.41,4168.72,6.19
